# Notebook 1 : Préparation des données et agrégation des annotations

Ce notebook prépare le dataset brut pour l’entraînement et l’évaluation du modèle.

Le dataset initial contient des tweets annotés par plusieurs annotateurs. Chaque annotateur fournit un label et, lorsque c’est pertinent, une proposition implicite manquante.

But : Donner pour le modèle dans les notebook ultérieurs pas seulement une information catégorielle, mais aussi l'information sur le niveau d'incertitude selon le niveau de d'accord entre les annotateurs 

Le dataset n'est pas présent sur GitHub par raisons de confidentialité. 

**Fichiers produits**

- data/processed/enthymemes_aggregated.json
- data/processed/enthymemes_aggregated.jsonl
- data/processed/train_aggregated.jsonl
- data/processed/validation_aggregated.jsonl
- data/processed/test_aggregated.jsonl

Ces fichiers sont ensuite utilisés pour créer le dataset de supervised fine-tuning.



### Chargement

In [1]:
import json
import random
from pathlib import Path
from collections import Counter
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd

In [ ]:
# Chemins 

#deuxieme version du fichier d'annotations 
INPUT_PATH = Path("/Users/quentinnippert/Documents/projet_python/enthymemes_2/merged_annotations_v2.json")

# Outputs
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Splits 
TRAIN_SIZE = 0.8
VAL_SIZE = 0.1
TEST_SIZE = 0.1
SEED = 42

VALID_LABELS = ["none", "premise", "claim"]
ALL_LABELS = ["none", "premise", "claim", "ambiguous"]

In [ ]:
#Chargement de JSON et vérification du format

def load_json_dataset(path: Path) -> List[Dict[str, Any]]:
    """
    Загружает JSON.

    Поддерживает:
    1. Корень файла = list
    2. Корень файла = dict с ключом data/items/examples/records
    """

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ["data", "items", "examples", "records"]:
            if key in data and isinstance(data[key], list):
                return data[key]

    raise ValueError(
        "Unsupported JSON format. Expected a list or a dict with key: data/items/examples/records."
    )

In [42]:
raw_data = load_json_dataset(INPUT_PATH)

print("Number of tweets:", len(raw_data))
print("Type of first item:", type(raw_data[0]))
print("Keys of first item:", raw_data[0].keys())

raw_data[0]

Number of tweets: 1333
Type of first item: <class 'dict'>
Keys of first item: dict_keys(['id', 'tweet_text', 'annotations', 'majority_label'])


{'id': '0',
 'tweet_text': 'Andrew Giuliani says that the state needs to be held responsible for any employee who was coerced to get the vaccine and had adverse effects.',
 'annotations': [{'label': 'none', 'implicit_text': ''},
  {'label': 'none', 'implicit_text': ''},
  {'label': 'none', 'implicit_text': ''},
  {'label': 'none', 'implicit_text': ''},
  {'label': 'none', 'implicit_text': ''}],
 'majority_label': 'none'}

### Fonctions de normalisation 

In [ ]:


def clean_text(text: Any) -> str:
    """
    Убирает лишние пробелы.
    """

    if text is None:
        return ""

    text = str(text)
    text = " ".join(text.split())
    return text.strip()

In [44]:
def normalize_label(label: Any) -> str:
    """
    Приводит label к одному из трёх классов:
    none / premise / claim

    В твоём датасете:
    conclusion = claim
    """

    if label is None:
        return "none"

    label = str(label).strip().lower()

    mapping = {
        # no implicit
        "none": "none",
        "no": "none",
        "no implicit": "none",
        "no_implicit": "none",
        "neither": "none",
        "nan": "none",
        "": "none",

        # implicit premise
        "premise": "premise",
        "implicit premise": "premise",
        "implicit_premise": "premise",
        "missing premise": "premise",
        "missing_premise": "premise",

        # implicit claim / conclusion
        "claim": "claim",
        "implicit claim": "claim",
        "implicit_claim": "claim",
        "missing claim": "claim",
        "missing_claim": "claim",

        "conclusion": "claim",
        "implicit conclusion": "claim",
        "implicit_conclusion": "claim",
        "missing conclusion": "claim",
        "missing_conclusion": "claim",
    }

    if label not in mapping:
        raise ValueError(f"Unknown label: {label}")

    return mapping[label]

In [45]:
def get_first_existing_key(item: Dict[str, Any], possible_keys: List[str]) -> Optional[str]:
    """
    Ищет первый существующий ключ из списка.
    Нужно, если в JSON поля называются немного иначе.
    """

    for key in possible_keys:
        if key in item:
            return key

    return None

In [46]:
def extract_tweet_text(item: Dict[str, Any]) -> str:
    """
    Достаёт текст твита из возможных названий поля.
    """

    possible_keys = [
        "tweet_text",
        "tweet",
        "text",
        "content",
        "original_text"
    ]

    key = get_first_existing_key(item, possible_keys)

    if key is None:
        raise KeyError(f"Cannot find tweet text field. Available keys: {list(item.keys())}")

    return clean_text(item[key])

In [47]:
def extract_annotations(item: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Достаёт список аннотаций.

    Ожидаемый формат:
    "annotations": [
        {"label": "...", "implicit_text": "..."},
        ...
    ]
    """

    possible_keys = [
        "annotations",
        "annotators",
        "annotation",
        "labels"
    ]

    key = get_first_existing_key(item, possible_keys)

    if key is None:
        raise KeyError(f"Cannot find annotations field. Available keys: {list(item.keys())}")

    annotations = item[key]

    if not isinstance(annotations, list):
        raise ValueError(f"Annotations field must be a list, got {type(annotations)}")

    return annotations

### Fonctions d'aggregation 

Après avoir normalisé les annotations individuelles, chaque tweet est transformé en un enregistrement agrégé.

L’objectif est de passer d’un niveau annotateur à un niveau tweet.

Chaque tweet reçoit donc plusieurs informations : le texte original, les labels normalisés, la distribution des votes, le label majoritaire, le niveau d’incertitude et les propositions implicites disponibles.

| Clé | Signification | Pourquoi cette information est utile |
|---|---|---|
| `id` | Identifiant unique du tweet. | Permet de suivre chaque tweet dans tout le pipeline : préparation, entraînement, inférence et évaluation. |
| `tweet_text` | Texte original du tweet. | C’est l’entrée principale du modèle. Le modèle apprend à partir de ce texte. |
| `n_annotators` | Nombre d’annotateurs disponibles pour ce tweet. | Permet de savoir sur combien d’annotations repose l’agrégation. Dans notre cas, il y a généralement cinq annotateurs par tweet. |
| `annotations_normalized` | Liste des annotations après normalisation des labels. | Permet de conserver les annotations individuelles dans un format homogène. Par exemple, `conclusion` est normalisé en `claim`. |
| `label_counts` | Nombre de votes pour chaque label : `none`, `premise`, `claim`. | Sert à calculer le label majoritaire et à mesurer le degré d’accord entre les annotateurs. |
| `vote_distribution` | Distribution proportionnelle des votes entre les labels. | Permet de représenter le désaccord entre annotateurs. Par exemple, `{none: 0.2, premise: 0.4, claim: 0.4}` indique un cas très incertain. |
| `majority_label` | Label ayant reçu le plus grand nombre de votes. | Sert de label principal pour la classification. Il peut être `none`, `premise`, `claim` ou `ambiguous` lorsqu’il n’y a pas de majorité claire. |
| `implicit_status` | Indique si le tweet contient ou non une argumentation implicite. | Simplifie l’information pour distinguer les tweets sans enthymème (`none`) des tweets contenant une proposition implicite (`premise` ou `claim`). |
| `confidence` | Score de confiance calculé à partir de l’accord entre annotateurs. | Plus le score est élevé, plus les annotateurs sont d’accord. Par exemple, 5 votes sur 5 donnent une confiance de `1.0`. |
| `uncertainty_level` | Niveau d’incertitude : `low`, `medium` ou `high`. | Permet d’intégrer le désaccord annotateur dans le dataset. Le modèle peut ainsi apprendre à prédire non seulement un label, mais aussi un degré d’incertitude. |
| `has_label_tie` | Indique s’il existe une égalité entre plusieurs labels. | Permet d’identifier les cas où il n’y a pas de majorité claire. Ces cas sont marqués comme `ambiguous`. |
| `tied_labels` | Liste des labels impliqués dans une égalité. | Permet de savoir entre quels labels les annotateurs sont divisés. |
| `valid_implicit_texts` | Liste de toutes les propositions implicites non vides fournies par les annotateurs. | Sert à conserver toutes les formulations possibles d’une proposition implicite. Cette information est utile pour la génération et pour l’évaluation multi-références. |
| `majority_implicit_texts` | Propositions implicites associées au label majoritaire. | Permet de garder seulement les formulations correspondant au label considéré comme principal. |
| `canonical_implicit_text` | Proposition implicite principale sélectionnée automatiquement. | Sert de référence principale pour l’entraînement et l’évaluation de la génération. Ce n’est pas une vérité absolue, mais une référence technique choisie parmi les annotations disponibles. |
| `usable_for_classification` | Indique si le tweet peut être utilisé comme exemple fiable pour la classification. | Cette valeur est `True` uniquement si le tweet a un label majoritaire stable parmi `none`, `premise` ou `claim`, sans égalité entre labels. |
| `usable_for_generation` | Indique si le tweet peut être utilisé pour la génération. | Cette valeur est `True` lorsqu’au moins une proposition implicite non vide a été fournie par un annotateur. |

In [48]:
def compute_majority_label(
    label_counts: Dict[str, int],
) -> Tuple[str, bool, List[str]]:
    """
    Возвращает:
    - majority_label
    - tie
    - tied_labels

    Если есть tie, majority_label = "ambiguous".
    """

    max_votes = max(label_counts.values())

    tied_labels = [
        label
        for label, count in label_counts.items()
        if count == max_votes
    ]

    tie = len(tied_labels) > 1

    if tie:
        return "ambiguous", True, tied_labels

    return tied_labels[0], False, tied_labels

In [49]:
def compute_uncertainty(confidence: float, tie: bool) -> str:
    """
    Для 5 аннотаторов:
    5/5 или 4/5 -> low
    3/5 -> medium
    2/5 или tie -> high
    """

    if tie:
        return "high"

    if confidence >= 0.8:
        return "low"

    if confidence >= 0.6:
        return "medium"

    return "high"

In [50]:
def deduplicate_texts(texts: List[str]) -> List[str]:
    """
    Убирает дубликаты implicit_text, сохраняя порядок.
    """

    seen = set()
    result = []

    for text in texts:
        normalized = " ".join(text.lower().split())

        if normalized and normalized not in seen:
            seen.add(normalized)
            result.append(text)

    return result

In [51]:
def select_canonical_implicit_text(texts: List[str]) -> str:
    """
    Выбирает один implicit_text для full-pipeline примера.

    Это не значит, что остальные варианты теряются:
    все варианты сохраняются в majority_implicit_texts.
    """

    texts = [text for text in texts if text.strip()]

    if not texts:
        return ""

    normalized_texts = [
        " ".join(text.lower().split())
        for text in texts
    ]

    counts = Counter(normalized_texts)
    most_common_normalized, _ = counts.most_common(1)[0]

    for original_text, normalized_text in zip(texts, normalized_texts):
        if normalized_text == most_common_normalized:
            return original_text

    return texts[0]

### Traitement d'un tweet 

In [52]:
def process_item(item: Dict[str, Any], index: int) -> Dict[str, Any]:
    """
    Превращает один твит с 5 аннотациями в агрегированную запись.
    """

    tweet_id = str(item.get("id", item.get("tweet_id", index)))
    tweet_text = extract_tweet_text(item)

    if not tweet_text:
        raise ValueError(f"Empty tweet text for item id={tweet_id}")

    raw_annotations = extract_annotations(item)

    normalized_annotations = []

    for ann_index, ann in enumerate(raw_annotations):
        if not isinstance(ann, dict):
            raise ValueError(f"Annotation must be dict, got {type(ann)} for tweet id={tweet_id}")

        label_key = get_first_existing_key(
            ann,
            [
                "label",
                "implicit_type",
                "type",
                "classification",
                "category"
            ]
        )

        implicit_text_key = get_first_existing_key(
            ann,
            [
                "implicit_text",
                "implicit",
                "implicit_argument",
                "missing_argument",
                "missing_proposition",
                "text"
            ]
        )

        raw_label = ann[label_key] if label_key is not None else "none"
        raw_implicit_text = ann[implicit_text_key] if implicit_text_key is not None else ""

        label = normalize_label(raw_label)
        implicit_text = clean_text(raw_implicit_text)

        if label == "none":
            implicit_text = ""

        normalized_annotations.append(
            {
                "annotator_id": ann.get("annotator_id", ann_index),
                "label": label,
                "implicit_text": implicit_text,
            }
        )

    labels = [ann["label"] for ann in normalized_annotations]
    n_annotators = len(labels)

    if n_annotators == 0:
        raise ValueError(f"No annotations found for tweet id={tweet_id}")

    counter = Counter(labels)

    label_counts = {
        label: counter.get(label, 0)
        for label in VALID_LABELS
    }

    majority_label, tie, tied_labels = compute_majority_label(
        label_counts=label_counts
    )

    if majority_label == "ambiguous":
        max_votes = max(label_counts.values())
    else:
        max_votes = label_counts[majority_label]

    confidence = max_votes / n_annotators

    vote_distribution = {
        label: round(label_counts[label] / n_annotators, 4)
        for label in VALID_LABELS
    }

    uncertainty_level = compute_uncertainty(confidence, tie)

    valid_implicit_texts = [
        ann["implicit_text"]
        for ann in normalized_annotations
        if ann["label"] != "none" and ann["implicit_text"]
    ]

    if majority_label == "ambiguous":
        majority_implicit_texts = []
    else:
        majority_implicit_texts = [
            ann["implicit_text"]
            for ann in normalized_annotations
            if ann["label"] == majority_label and ann["implicit_text"]
        ]

    valid_implicit_texts = deduplicate_texts(valid_implicit_texts)
    majority_implicit_texts = deduplicate_texts(majority_implicit_texts)

    canonical_implicit_text = select_canonical_implicit_text(majority_implicit_texts)

    if majority_label == "none":
        implicit_status = "no"
        canonical_implicit_text = ""

    elif majority_label == "ambiguous":
        implicit_status = "ambiguous"
        canonical_implicit_text = ""

    else:
        implicit_status = "yes"
    return {
        "id": tweet_id,
        "tweet_text": tweet_text,

        "n_annotators": n_annotators,
        "annotations_normalized": normalized_annotations,

        "label_counts": label_counts,
        "vote_distribution": vote_distribution,

        "majority_label": majority_label,
        "implicit_status": implicit_status,

        "confidence": round(confidence, 4),
        "uncertainty_level": uncertainty_level,
        "has_label_tie": tie,
        "tied_labels": tied_labels,

        "valid_implicit_texts": valid_implicit_texts,
        "majority_implicit_texts": majority_implicit_texts,
        "canonical_implicit_text": canonical_implicit_text,

        "usable_for_classification": majority_label in ["none", "premise", "claim"] and not tie,
        "usable_for_generation": len(valid_implicit_texts) > 0,
    }

### Application de preprocessing pour le dataset en entier

In [53]:
processed_records = []

errors = []

for index, item in enumerate(raw_data):
    try:
        processed_item = process_item(item, index)
        processed_records.append(processed_item)
    except Exception as e:
        errors.append(
            {
                "index": index,
                "error": str(e),
                "item_keys": list(item.keys()) if isinstance(item, dict) else None,
            }
        )

print("Successfully processed:", len(processed_records))
print("Errors:", len(errors))

Successfully processed: 1333
Errors: 0


In [ ]:
# EXEMPLE
processed_records[1]

{'id': '1',
 'tweet_text': "Keith Muise, a 41-year-old man in Newfoundland, says it's absurd he can't access a fourth dose of a COVID-19 vaccine despite the emergence of new ...Calls mount in N.L. for access to 4th COVID-19 shots | CBC News",
 'n_annotators': 5,
 'annotations_normalized': [{'annotator_id': 0,
   'label': 'premise',
   'implicit_text': 'new variants justify access to additional vaccine doses'},
  {'annotator_id': 1,
   'label': 'premise',
   'implicit_text': 'You should be able to get an extra shot of a vaccine if there is a new variant'},
  {'annotator_id': 2,
   'label': 'premise',
   'implicit_text': 'the emergence of new covid variants are not covered by the old covid vaccines, but the new caccines do cover them'},
  {'annotator_id': 3,
   'label': 'premise',
   'implicit_text': 'New variants justify access to additional vaccine doses'},
  {'annotator_id': 4,
   'label': 'premise',
   'implicit_text': 'You should be able to get an extra shot of a vaccine if there is

### Stats de dataset

In [55]:
df = pd.DataFrame(processed_records)

df.head(10)

,id,tweet_text,n_annotators,annotations_normalized,label_counts,vote_distribution,majority_label,implicit_status,confidence,uncertainty_level,has_label_tie,tied_labels,valid_implicit_texts,majority_implicit_texts,canonical_implicit_text,usable_for_classification,usable_for_generation
0,0,Andrew Giuliani says that the state needs to b...,5,"[{'annotator_id': 0, 'label': 'none', 'implici...","{'none': 5, 'premise': 0, 'claim': 0}","{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",none,no,1.0,low,False,[none],[],[],,True,False
1,1,"Keith Muise, a 41-year-old man in Newfoundland...",5,"[{'annotator_id': 0, 'label': 'premise', 'impl...","{'none': 0, 'premise': 5, 'claim': 0}","{'none': 0.0, 'premise': 1.0, 'claim': 0.0}",premise,yes,1.0,low,False,[premise],[new variants justify access to additional vac...,[new variants justify access to additional vac...,new variants justify access to additional vacc...,True,True
2,2,"NGO, Global Citizen provided World with materi...",5,"[{'annotator_id': 0, 'label': 'none', 'implici...","{'none': 5, 'premise': 0, 'claim': 0}","{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",none,no,1.0,low,False,[none],[],[],,True,False
3,3,"To go after Special Olympians, who all they wa...",5,"[{'annotator_id': 0, 'label': 'none', 'implici...","{'none': 5, 'premise': 0, 'claim': 0}","{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",none,no,1.0,low,False,[none],[],[],,True,False
4,5,Coronavirus vaccines may not prevent many symp...,5,"[{'annotator_id': 0, 'label': 'none', 'implici...","{'none': 5, 'premise': 0, 'claim': 0}","{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",none,no,1.0,low,False,[none],[],[],,True,False
5,6,You bigot. Isn't this exactly what you did wit...,5,"[{'annotator_id': 0, 'label': 'none', 'implici...","{'none': 5, 'premise': 0, 'claim': 0}","{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",none,no,1.0,low,False,[none],[],[],,True,False
6,7,Nmo?????? Covid-19 vaccines prevented over 42 ...,5,"[{'annotator_id': 0, 'label': 'none', 'implici...","{'none': 5, 'premise': 0, 'claim': 0}","{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",none,no,1.0,low,False,[none],[],[],,True,False
7,8,"He's correct, the left destroyed, 'My body my ...",5,"[{'annotator_id': 0, 'label': 'premise', 'impl...","{'none': 2, 'premise': 3, 'claim': 0}","{'none': 0.4, 'premise': 0.6, 'claim': 0.0}",premise,yes,0.6,medium,False,[premise],[the left cannot claim bodily autonomy after s...,[the left cannot claim bodily autonomy after s...,the left cannot claim bodily autonomy after su...,True,True
8,9,NYC's monkeypox vaccine rollout halted by low ...,5,"[{'annotator_id': 0, 'label': 'none', 'implici...","{'none': 5, 'premise': 0, 'claim': 0}","{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",none,no,1.0,low,False,[none],[],[],,True,False
9,11,I thought for a moment this couldn't possibly ...,5,"[{'annotator_id': 0, 'label': 'premise', 'impl...","{'none': 0, 'premise': 4, 'claim': 1}","{'none': 0.0, 'premise': 0.8, 'claim': 0.2}",premise,yes,0.8,low,False,[premise],[wide teaching CPR and widespread availability...,[wide teaching CPR and widespread availability...,wide teaching CPR and widespread availability ...,True,True


In [56]:
print("Total tweets:", len(df))

print("\nMajority label distribution:")
display(df["majority_label"].value_counts())

print("\nImplicit status distribution:")
display(df["implicit_status"].value_counts())

print("\nUncertainty distribution:")
display(df["uncertainty_level"].value_counts())

print("\nUsable for generation:")
display(df["usable_for_generation"].value_counts())

print("\nLabel ties:")
display(df["has_label_tie"].value_counts())

Total tweets: 1333

Majority label distribution:


majority_label
none         882
premise      371
claim         48
ambiguous     32
Name: count, dtype: int64


Implicit status distribution:


implicit_status
no           882
yes          419
ambiguous     32
Name: count, dtype: int64


Uncertainty distribution:


uncertainty_level
low       1153
medium     148
high        32
Name: count, dtype: int64


Usable for generation:


usable_for_generation
True     726
False    607
Name: count, dtype: int64


Label ties:


has_label_tie
False    1301
True       32
Name: count, dtype: int64

In [57]:
# Более удобная таблица
summary = pd.DataFrame({
    "majority_label": df["majority_label"].value_counts(),
})

summary

,majority_label
majority_label,
none,882
premise,371
claim,48
ambiguous,32


### Splits

In [58]:
def split_by_tweets(
    records: List[Dict[str, Any]],
    train_size: float = 0.8,
    val_size: float = 0.1,
    test_size: float = 0.1,
    seed: int = 42,
) -> Dict[str, List[Dict[str, Any]]]:
    """
    Делит данные по твитам.
    """

    assert abs(train_size + val_size + test_size - 1.0) < 1e-6

    records = records.copy()

    random.seed(seed)
    random.shuffle(records)

    n = len(records)

    n_train = int(n * train_size)
    n_val = int(n * val_size)

    train_records = records[:n_train]
    val_records = records[n_train:n_train + n_val]
    test_records = records[n_train + n_val:]

    for record in train_records:
        record["split"] = "train"

    for record in val_records:
        record["split"] = "validation"

    for record in test_records:
        record["split"] = "test"

    return {
        "train": train_records,
        "validation": val_records,
        "test": test_records,
    }

In [59]:
split_data = split_by_tweets(
    processed_records,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    test_size=TEST_SIZE,
    seed=SEED,
)

train_records = split_data["train"]
validation_records = split_data["validation"]
test_records = split_data["test"]

print("Train:", len(train_records))
print("Validation:", len(validation_records))
print("Test:", len(test_records))

Train: 1066
Validation: 133
Test: 134


In [60]:
def split_stats(records: List[Dict[str, Any]], name: str) -> pd.DataFrame:
    temp_df = pd.DataFrame(records)

    stats = {
        "split": name,
        "n_tweets": len(temp_df),
        "n_none": int((temp_df["majority_label"] == "none").sum()),
        "n_premise": int((temp_df["majority_label"] == "premise").sum()),
        "n_claim": int((temp_df["majority_label"] == "claim").sum()),
        "n_low_uncertainty": int((temp_df["uncertainty_level"] == "low").sum()),
        "n_medium_uncertainty": int((temp_df["uncertainty_level"] == "medium").sum()),
        "n_high_uncertainty": int((temp_df["uncertainty_level"] == "high").sum()),
        "n_usable_for_generation": int(temp_df["usable_for_generation"].sum()),
    }

    return pd.DataFrame([stats])

In [61]:
stats_df = pd.concat(
    [
        split_stats(train_records, "train"),
        split_stats(validation_records, "validation"),
        split_stats(test_records, "test"),
    ],
    ignore_index=True,
)

stats_df

,split,n_tweets,n_none,n_premise,n_claim,n_low_uncertainty,n_medium_uncertainty,n_high_uncertainty,n_usable_for_generation
0,train,1066,708,294,39,919,122,25,578
1,validation,133,87,40,4,119,12,2,70
2,test,134,87,37,5,115,14,5,78


### Enregistrement de fichiers

In [62]:
def write_jsonl(records: List[Dict[str, Any]], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def write_json(records: List[Dict[str, Any]], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

In [63]:
all_records_with_split = train_records + validation_records + test_records

write_json(
    all_records_with_split,
    OUTPUT_DIR / "enthymemes_aggregated.json"
)

write_jsonl(
    all_records_with_split,
    OUTPUT_DIR / "enthymemes_aggregated.jsonl"
)

write_jsonl(
    train_records,
    OUTPUT_DIR / "train_aggregated.jsonl"
)

write_jsonl(
    validation_records,
    OUTPUT_DIR / "validation_aggregated.jsonl"
)

write_jsonl(
    test_records,
    OUTPUT_DIR / "test_aggregated.jsonl"
)

print("Saved files:")
print(OUTPUT_DIR / "enthymemes_aggregated.json")
print(OUTPUT_DIR / "enthymemes_aggregated.jsonl")
print(OUTPUT_DIR / "train_aggregated.jsonl")
print(OUTPUT_DIR / "validation_aggregated.jsonl")
print(OUTPUT_DIR / "test_aggregated.jsonl")

Saved files:
data/processed/enthymemes_aggregated.json
data/processed/enthymemes_aggregated.jsonl
data/processed/train_aggregated.jsonl
data/processed/validation_aggregated.jsonl
data/processed/test_aggregated.jsonl


In [66]:
example = all_records_with_split[3]

print("Tweet:")
print(example["tweet_text"])

print("\nMajority label:")
print(example["majority_label"])

print("\nImplicit status:")
print(example["implicit_status"])

print("\nVote distribution:")
print(example["vote_distribution"])

print("\nLabel counts:")
print(example["label_counts"])

print("\nConfidence:")
print(example["confidence"])

print("\nUncertainty:")
print(example["uncertainty_level"])

print("\nHas label tie:")
print(example["has_label_tie"])

print("\nTied labels:")
print(example.get("tied_labels", []))

print("\nUsable for classification:")
print(example["usable_for_classification"])

print("\nUsable for generation:")
print(example["usable_for_generation"])

print("\nCanonical implicit text:")
print(example["canonical_implicit_text"])

print("\nAll majority implicit texts:")
print(example["majority_implicit_texts"])

print("\nAll valid implicit texts:")
print(example["valid_implicit_texts"])

print("\nNormalized annotations:")
for ann in example["annotations_normalized"]:
    print(f"- annotator {ann['annotator_id']} | label: {ann['label']} | implicit_text: {ann['implicit_text']}")

Tweet:
#AsiaBibi's acquittal has been upheld by Pakistan's supreme court. She is still in fear for her life however. Remember that the UK refused this Christian woman asylum because of fears it would cause "unrest". An indelible disgrace on our government

Majority label:
ambiguous

Implicit status:
ambiguous

Vote distribution:
{'none': 0.2, 'premise': 0.4, 'claim': 0.4}

Label counts:
{'none': 1, 'premise': 2, 'claim': 2}

Confidence:
0.4

Uncertainty:
high

Has label tie:
True

Tied labels:
['premise', 'claim']

Usable for classification:
False

Usable for generation:
True

Canonical implicit text:


All majority implicit texts:
[]

All valid implicit texts:
['A government that refuses asylum to a persecuted person due to fears of domestic unrest is acting disgracefully.', 'The UK should have granted this woman asylum', 'The UK should have granted Asia Bibi asylum']

Normalized annotations:
- annotator 0 | label: premise | implicit_text: A government that refuses asylum to a persecu